# Predicting Surface Roughness in a Machining Process

This project compares two regression models — **Random Forest** and **K-Nearest Neighbors** —
for predicting surface roughness (`Avg. Ra`) of a workpiece from three process parameters:

- **TS** — Traverse Speed (mm/min)
- **OCV** — Open Circuit Voltage (V)
- **WFR** — Wire Feed Rate (m/min)

The goal is to see which model better captures the (nonlinear) relationship between
machining parameters and surface finish, using a small experimental dataset (27 runs).

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error

RANDOM_STATE = 45  # single fixed seed used everywhere for reproducibility

## 2. Load and explore the data

In [ ]:
data = pd.read_excel("explodata.xlsx")
data.head()

In [ ]:
data.describe()

In [ ]:
data.info()

**Exploratory data analysis.** With only 27 rows this won't reveal much statistically,
but it's worth checking for obvious relationships and outliers before modeling.

In [ ]:
corr = data.drop(columns=["Sl No"]).corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation between process parameters and surface roughness")
plt.tight_layout()
import os
os.makedirs("images", exist_ok=True)
plt.savefig("images/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["TS (mm/min)", "OCV (v)", "WFR (m/min)"]):
    ax.scatter(data[col], data["Avg. Ra (µm)"])
    ax.set_xlabel(col)
    ax.set_ylabel("Avg. Ra (µm)")
plt.suptitle("Surface roughness vs. each process parameter")
plt.tight_layout()
plt.savefig("images/feature_vs_target.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Train / test split

We split once and reuse the same split for both models, so their scores are directly comparable.

In [ ]:
FEATURES = ["TS (mm/min)", "OCV (v)", "WFR (m/min)"]
TARGET = "Avg. Ra (µm)"

X = data[FEATURES]
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
X_train.shape, X_test.shape

## 4. Random Forest Regressor

Random Forest doesn't need feature scaling, so we train it directly on the raw features.

In [ ]:
rf_baseline = RandomForestRegressor(n_estimators=30, random_state=RANDOM_STATE)
rf_baseline.fit(X_train, y_train)
rf_baseline_pred = rf_baseline.predict(X_test)

print("Baseline Random Forest R2:", r2_score(y_test, rf_baseline_pred))

**Hyperparameter tuning with grid search.** With only ~21 training rows, we use 3-fold CV to keep folds a reasonable size.

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 3, 5, 10],
    "min_samples_split": [2, 4],
    "min_samples_leaf": [1, 2],
}

grid = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
rf = grid.best_estimator_

In [ ]:
rf_pred = rf.predict(X_test)

rf_r2 = r2_score(y_test, rf_pred)
rf_mse = mean_squared_error(y_test, rf_pred)
print(f"Tuned Random Forest -> R2: {rf_r2:.3f}, MSE: {rf_mse:.4f}")

## 5. K-Nearest Neighbors Regressor

KNN is distance-based, so we scale features using the **same train/test split** as the Random Forest model.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Choosing k.** A common rule of thumb is `k ≈ sqrt(n_train)`, but with this few samples
it's more reliable to just try a small range of k values directly.

In [ ]:
k_candidates = range(1, 10)
k_scores = []

for k in k_candidates:
    knn_k = KNeighborsRegressor(n_neighbors=k)
    knn_k.fit(X_train_scaled, y_train)
    pred_k = knn_k.predict(X_test_scaled)
    k_scores.append(r2_score(y_test, pred_k))

plt.figure(figsize=(6, 4))
plt.plot(list(k_candidates), k_scores, marker="o")
plt.xlabel("k (number of neighbors)")
plt.ylabel("R2 score on test set")
plt.title("KNN: choosing k")
plt.tight_layout()
plt.savefig("images/knn_k_selection.png", dpi=150, bbox_inches="tight")
plt.show()

best_k = list(k_candidates)[int(np.argmax(k_scores))]
print("Best k:", best_k)

In [ ]:
knn = KNeighborsRegressor(n_neighbors=best_k)
knn.fit(X_train_scaled, y_train)
knn_pred = knn.predict(X_test_scaled)

knn_r2 = r2_score(y_test, knn_pred)
knn_mse = mean_squared_error(y_test, knn_pred)
print(f"Tuned KNN (k={best_k}) -> R2: {knn_r2:.3f}, MSE: {knn_mse:.4f}")

## 6. Model comparison

In [ ]:
print(f"Random Forest -> R2: {rf_r2:.3f}, MSE: {rf_mse:.4f}")
print(f"KNN            -> R2: {knn_r2:.3f}, MSE: {knn_mse:.4f}")

In [ ]:
x_axis = np.arange(len(y_test))

plt.figure(figsize=(8, 5))
plt.plot(x_axis, y_test.values, label="Actual", marker="o")
plt.plot(x_axis, rf_pred, label="Random Forest predicted", marker="o")
plt.plot(x_axis, knn_pred, label="KNN predicted", marker="o")
plt.xlabel("Test sample index")
plt.ylabel("Avg. Ra (µm)")
plt.title("Actual vs. Predicted Surface Roughness")
plt.legend()
plt.tight_layout()
plt.savefig("images/actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Predicting on a new set of process parameters

In [ ]:
new_params = pd.DataFrame([[250, 23, 6]], columns=FEATURES)

rf_new_pred = rf.predict(new_params)[0]
knn_new_pred = knn.predict(scaler.transform(new_params))[0]

print(f"Random Forest predicted Ra: {rf_new_pred:.3f} µm")
print(f"KNN predicted Ra:           {knn_new_pred:.3f} µm")

## 8. Conclusion

- Both models were evaluated on the **same train/test split** and scored with `r2_score(y_test, y_pred)`
  so the comparison is apples-to-apples (the original version of this notebook had the R² arguments
  reversed in places, which silently produced misleading scores).
- Random Forest hyperparameters were tuned with grid search + cross-validation instead of hand-picked values.
- KNN's `k` was chosen by evaluating a range of values rather than a fixed guess, and features were
  scaled consistently before every KNN fit/predict call.
- **Caveat:** the dataset has only 27 experimental runs, so these results should be read as a
  proof-of-concept comparison rather than a statistically robust conclusion. A larger design-of-experiments
  dataset would be needed to draw strong conclusions about which model generalizes better.